# 🎮 Craftiverse — Consumer Behaviour Analysis
### Factions & Skyblock Network · Powered by SQLite + Pandas + Seaborn

> **Objective:** Translate raw player-activity data into clear, monetisation-ready business decisions.

| Item | Value |
|---|----|
| Database | `craftiverse.db` |
| Players | 580 |
| Store Products | 19 |
| Analysis date | 2026-05-22 |

---
**Sections**
1. [Data Ingestion](#step-1)
2. [KPI Cheatsheet](#kpi-sql)
3. [Part A — Player Psychology](#part-a)
4. [Part B — Targeted Segments](#part-b)
5. [Part C — Server Hype](#part-c)
6. [Part D — Financials](#part-d)
7. [Part E — Growth & New Products](#part-e)
8. [README Generation](#readme)
9. [PowerBI Export](#powerbi)
10. [Revenue Growth Summary](#revenue-growth)

In [ ]:
import sqlite3, json, warnings, textwrap
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

C = {
    'green':   '#2ecc71', 'blue':    '#3498db', 'red':     '#e74c3c',
    'gold':    '#f1c40f', 'purple':  '#9b59b6', 'orange':  '#e67e22',
    'teal':    '#1abc9c', 'dark_bg': '#1a1a2e', 'card_bg': '#16213e',
    'text':    '#ecf0f1', 'grid':    '#2c3e50',
}
RANK_COLORS = {'Default': '#95a5a6', 'VIP': '#2ecc71', 'MVP': '#3498db', 'Legend': '#f1c40f'}
CAT_COLORS  = {
    'Ranks': C['gold'], 'Perks': C['green'], 'Crate keys': C['purple'],
    'Sellwands': C['blue'], 'Gkits': C['orange'], 'Tags': C['teal'],
}

plt.rcParams.update({
    'figure.facecolor': C['dark_bg'], 'axes.facecolor': C['card_bg'],
    'text.color': C['text'], 'axes.labelcolor': C['text'],
    'xtick.color': C['text'], 'ytick.color': C['text'],
    'grid.color': C['grid'], 'grid.alpha': 0.5,
    'axes.titlesize': 16, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'font.size': 12, 'legend.fontsize': 11,
    'legend.facecolor': '#0f3460', 'legend.edgecolor': '#444',
    'legend.labelcolor': C['text'],
    'axes.spines.top': False, 'axes.spines.right': False,
})

def style(ax, title, xlabel='', ylabel='', rotate_x=False):
    ax.set_title(title, pad=14, color=C['text'], fontsize=16, fontweight='bold')
    ax.set_xlabel(xlabel, color=C['text'], fontsize=13, labelpad=8)
    ax.set_ylabel(ylabel, color=C['text'], fontsize=13, labelpad=8)
    if rotate_x:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')

DB_PATH = Path('craftiverse.db')
print(f"★  Environment ready | {datetime.now().strftime('%Y-%m-%d %H:%M')}")


<a id="step-1"></a>
## Step 1 — Data Ingestion

Connect to `craftiverse.db` and load both tables into Pandas DataFrames.
The JSON columns (`purchased_items_list`, `cart_items_list`) are parsed into
Python lists and immediately normalised into two flat fact tables that power
every downstream analysis.

**Schema recap**

| Table | Key columns |
|----|-------|
| `players_data` | id, username, rank, first_join_date, total_playtime_hours, total_spent_dollars, total_votes, webstore_visits, last_purchase_date, total_transactions, cart_abandonments, purchased_items_list, cart_items_list |
| `store_products` | id, category, product_name, price |

In [ ]:
# -- Load raw tables -------------------------------------------------------
def load_raw():
    conn = sqlite3.connect(DB_PATH)
    pl = pd.read_sql_query("SELECT * FROM players_data",  conn)
    pr = pd.read_sql_query("SELECT * FROM store_products", conn)
    conn.close()
    return pl, pr

players, products = load_raw()

# -- Date parsing ----------------------------------------------------------
for col in ['first_join_date', 'last_purchase_date']:
    players[col] = pd.to_datetime(players[col], errors='coerce')

# -- JSON parsing ----------------------------------------------------------
def safe_json(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', "'{}'", 'null'):
        return []
    try:
        return json.loads(val)
    except Exception:
        return []

players['purchased_items_list'] = players['purchased_items_list'].apply(safe_json)
players['cart_items_list']      = players['cart_items_list'].apply(safe_json)

# -- Derived columns -------------------------------------------------------
players['cart_total_value'] = players['cart_items_list'].apply(
    lambda lst: sum(i.get('price', 0) for i in lst)
)
players['is_payer'] = players['total_spent_dollars'] > 0

# -- Flat fact tables ------------------------------------------------------
def explode_items(df, col):
    rows = []
    for _, r in df.iterrows():
        for item in r[col]:
            rows.append({
                'player_id': r['id'],
                'username':  r['username'],
                'rank':      r['rank'],
                'item':      item.get('item', ''),
                'price':     item.get('price', 0),
            })
    if not rows:
        return pd.DataFrame(columns=['player_id','username','rank','item','price'])
    return pd.DataFrame(rows)

purchases_flat = explode_items(players, 'purchased_items_list').merge(
    products[['product_name','category']],
    left_on='item', right_on='product_name', how='left'
).drop(columns='product_name')

cart_flat = explode_items(players, 'cart_items_list').merge(
    products[['product_name','category']],
    left_on='item', right_on='product_name', how='left'
).drop(columns='product_name')

# -- Summary ---------------------------------------------------------------
print(f"{'-'*55}")
print(f"  players_data      : {len(players):>6,} rows")
print(f"  store_products    : {len(products):>6,} rows")
print(f"  purchases (flat)  : {len(purchases_flat):>6,} rows")
print(f"  cart items (flat) : {len(cart_flat):>6,} rows")
print(f"{'-'*55}")
print(f"  Paying players    : {players['is_payer'].sum():>6,}  ({players['is_payer'].mean()*100:.1f}%)")
print(f"  Total Revenue     : ${players['total_spent_dollars'].sum():>9,.2f}")
print(f"  Abandoned value   : ${players['cart_total_value'].sum():>9,.2f}")
print(f"{'-'*55}")
players.head(3)


<a id="kpi-sql"></a>
## KPI Cheatsheet — SQL Deep Dive

The six queries below are executed **directly from `kpi_cheatsheet.sql`** against the live database.
Each KPI answers a distinct business question and feeds into the downstream Part A–E analyses.

| # | KPI Name | Business Question |
|---|---|---|
| 1 | **Product Popularity** | Which specific products sell the most units? |
| 2 | **Category Revenue** | Which product category generates the most money? |
| 3 | **Cart Abandonment Leaderboard** | Which items are most often abandoned at checkout? |
| 4 | **User Conversion Overview** | What proportion of players actually buy something? |
| 5 | **Top 10 Spenders (Whales)** | Who are the highest-value individual customers? |
| 6 | **Rank Value** | Which server rank tier contributes the most revenue per-player and in total? |

> Key findings: Perks ($3,355) beat Ranks ($2,830) · Default rank = $6,072 total revenue · Abandoned cart ($13,125) **exceeds** actual revenue ($9,912)
>
> Charts saved as `kpi_overview_1.png` and `kpi_overview_2.png`

In [ ]:
# ============================================================
# KPI Cheatsheet — Direct SQL Execution from kpi_cheatsheet.sql
# ============================================================
import sqlite3

def _q(sql):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

kpi1 = _q("""SELECT json_extract(je.value,'$.item') AS product, COUNT(*) AS units
    FROM players_data pd, json_each(pd.purchased_items_list) je
    WHERE pd.purchased_items_list IS NOT NULL GROUP BY 1 ORDER BY 2 DESC""")

kpi2 = _q("""SELECT sp.category, COUNT(*) AS units, ROUND(SUM(sp.price),2) AS revenue
    FROM players_data pd, json_each(pd.purchased_items_list) je
    JOIN store_products sp ON sp.product_name = json_extract(je.value,'$.item')
    WHERE pd.purchased_items_list IS NOT NULL GROUP BY 1 ORDER BY 3 DESC""")

kpi3 = _q("""SELECT json_extract(je.value,'$.item') AS product, COUNT(*) AS abandoned
    FROM players_data pd, json_each(pd.cart_items_list) je
    WHERE pd.cart_items_list IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 5""")

kpi4 = _q("""SELECT CASE WHEN total_spent_dollars>0 THEN 'Paying' ELSE 'Non-Paying' END AS seg,
           COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),1) AS pct
    FROM players_data GROUP BY 1 ORDER BY 2 DESC""")

kpi5 = _q("""SELECT username, rank, ROUND(total_spent_dollars,2) AS spent,
           total_playtime_hours AS hrs, total_votes AS votes
    FROM players_data ORDER BY total_spent_dollars DESC LIMIT 10""")

kpi6 = _q("""SELECT rank, COUNT(*) AS players,
           ROUND(AVG(total_spent_dollars),2) AS avg_spent,
           ROUND(MAX(total_spent_dollars),2) AS max_spent,
           ROUND(SUM(total_spent_dollars),2) AS total_rev
    FROM players_data GROUP BY 1 ORDER BY 3 DESC""")

kpi_cart = _q("""SELECT ROUND(SUM(sp.price),2) AS abandoned_val, COUNT(*) AS items
    FROM players_data pd, json_each(pd.cart_items_list) je
    JOIN store_products sp ON sp.product_name = json_extract(je.value,'$.item')
    WHERE pd.cart_items_list IS NOT NULL""")

total_abandoned = kpi_cart['abandoned_val'].iloc[0]
total_revenue   = players['total_spent_dollars'].sum()
capture_rate    = total_revenue / (total_revenue + total_abandoned) * 100

# ── Figure 1: KPI 1 + KPI 2 + KPI 3 ──────────────────────────────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(28, 10), facecolor=C['dark_bg'])
fig1.suptitle('KPI Cheatsheet — Part 1: Sales Intelligence & Revenue Leak Detection',
              fontsize=18, fontweight='bold', color=C['text'], y=1.02)

# KPI 1 — Product Popularity
ax = axes1[0]
top10 = kpi1.head(10).sort_values('units')
bar_colors_1 = plt.cm.YlGn(np.linspace(0.45, 0.95, len(top10)))
bars_1 = ax.barh(top10['product'], top10['units'], color=bar_colors_1, edgecolor='#111', height=0.65)
for bar in bars_1:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())} units', va='center', ha='left',
            fontsize=11, color=C['text'], fontweight='bold')
ax.set_xlim(0, top10['units'].max() * 1.3)
ax.grid(axis='x', alpha=0.3, linewidth=0.8)
style(ax, 'KPI 1 — Product Popularity\nTop 10 Best-Selling Products', 'Units Sold', '')

# KPI 2 — Category Revenue
ax2 = axes1[1]
cat_colors_2 = [CAT_COLORS.get(c, C['blue']) for c in kpi2['category']]
bars_2 = ax2.bar(kpi2['category'], kpi2['revenue'], color=cat_colors_2, edgecolor='#111', width=0.6)
for bar, row in zip(bars_2, kpi2.itertuples()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'${row.revenue:,.0f}\n{row.units} units',
             ha='center', va='bottom', fontsize=10, color=C['text'],
             fontweight='bold', linespacing=1.5)
ax2.set_ylim(0, kpi2['revenue'].max() * 1.35)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax2, 'KPI 2 — Category Revenue\nTotal USD by Category', 'Category', 'Revenue (USD)', rotate_x=True)

# KPI 3 — Cart Abandonment (dramatic)
ax3 = axes1[2]
kpi3_s = kpi3.sort_values('abandoned')
red_grad = ['#e74c3c', '#c0392b', '#a93226', '#922b21', '#7b241c']
bars_3 = ax3.barh(kpi3_s['product'], kpi3_s['abandoned'],
                  color=red_grad, edgecolor='#111', height=0.6)
for bar in bars_3:
    ax3.text(bar.get_width() - 1, bar.get_y() + bar.get_height()/2,
             f'{int(bar.get_width())}×', va='center', ha='right',
             fontsize=14, color='white', fontweight='bold')
ax3.set_xlim(58, kpi3['abandoned'].max() * 1.2)
ax3.grid(axis='x', alpha=0.3, linewidth=0.8)
ax3.text(0.97, 0.04, f'⚠  ${total_abandoned:,.0f}\ntotal revenue\nleak detected',
         transform=ax3.transAxes, ha='right', va='bottom', fontsize=12,
         color=C['red'], fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', fc='#1a0505', ec=C['red'], alpha=0.9))
style(ax3, 'KPI 3 — Cart Abandonment Top 5\nMost-Abandoned Products', 'Times Abandoned', '')

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('kpi_overview_1.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

# ── Figure 2: KPI 4 + KPI 5 + KPI 6 ──────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(28, 10), facecolor=C['dark_bg'])
fig2.suptitle('KPI Cheatsheet — Part 2: Conversion, Whale Analysis & Rank Value',
              fontsize=18, fontweight='bold', color=C['text'], y=1.02)

# KPI 4 — Conversion Donut
ax4 = axes2[0]
pct_paying = kpi4[kpi4['seg']=='Paying']['pct'].values[0]
n_paying   = kpi4[kpi4['seg']=='Paying']['n'].values[0]
n_non      = kpi4[kpi4['seg']=='Non-Paying']['n'].values[0]
pct_non    = kpi4[kpi4['seg']=='Non-Paying']['pct'].values[0]
ax4.pie(kpi4['n'],
        labels=[f"Paying\n{n_paying} players\n({pct_paying}%)",
                f"Non-Paying\n{n_non} players\n({pct_non}%)"],
        colors=[C['green'], '#555'], startangle=90,
        wedgeprops={'width': 0.55, 'edgecolor': C['dark_bg'], 'linewidth': 3},
        textprops={'color': C['text'], 'fontsize': 12, 'fontweight': 'bold'})
ax4.text(0, 0, f'{pct_paying}%\nconversion', ha='center', va='center',
         fontsize=18, color=C['green'], fontweight='bold')
ax4.set_title('KPI 4 — User Conversion\nPaying vs. Non-Paying Players',
              color=C['text'], fontsize=16, fontweight='bold', pad=16)
ax4.text(0, -1.3, '★ 3-4× industry standard (20-30%)',
         ha='center', fontsize=11, color=C['gold'], fontweight='bold')

# KPI 5 — Top Whales
ax5 = axes2[1]
kpi5_s = kpi5.sort_values('spent')
w_col  = [RANK_COLORS.get(r, C['blue']) for r in kpi5_s['rank']]
bars_5 = ax5.barh(kpi5_s['username'], kpi5_s['spent'], color=w_col, edgecolor='#111', height=0.65)
for bar, row in zip(bars_5, kpi5_s.itertuples()):
    ax5.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'${row.spent:.0f}  [{row.rank}]', va='center',
             fontsize=10, color=C['text'], fontweight='bold')
ax5.set_xlim(0, kpi5['spent'].max() * 1.65)
ax5.grid(axis='x', alpha=0.3, linewidth=0.8)
rank_patches = [mpatches.Patch(color=v, label=k) for k, v in RANK_COLORS.items()]
ax5.legend(handles=rank_patches, fontsize=10, loc='lower right')
style(ax5, 'KPI 5 — Top 10 Spenders\n"Whale" Identification & Profiling', 'Lifetime Spend (USD)', '')

# KPI 6 — Rank Value
ax6 = axes2[2]
x6     = np.arange(len(kpi6))
w6     = 0.35
r_col6 = [RANK_COLORS.get(r, C['blue']) for r in kpi6['rank']]
b6a = ax6.bar(x6 - w6/2, kpi6['avg_spent'], w6, label='Avg Spend / Player',
              color=r_col6, edgecolor='#111', alpha=0.95)
b6b = ax6.bar(x6 + w6/2, kpi6['total_rev'] / 100, w6, label='Total Revenue (÷100 scale)',
              color=r_col6, edgecolor='#111', alpha=0.45, hatch='//')
ax6.bar_label(b6a, labels=[f'${v:.0f}' for v in kpi6['avg_spent']],
              fontsize=10, color=C['text'], padding=3)
ax6.bar_label(b6b, labels=[f'${v:,.0f}' for v in kpi6['total_rev']],
              fontsize=9, color=C['text'], padding=3)
ax6.set_xticks(x6)
ax6.set_xticklabels(kpi6['rank'], fontsize=12, color=C['text'])
ax6.legend(fontsize=10, loc='upper right')
ax6.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax6, 'KPI 6 — Rank Value\nAvg Spend & Total Revenue by Server Rank', 'Server Rank', 'USD ($)')

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('kpi_overview_2.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

print(f"{'═'*65}")
print(f"  KPI CHEATSHEET — LIVE RESULTS FROM craftiverse.db")
print(f"{'═'*65}")
print(f"  KPI 1  │ #1 Product    : {kpi1.iloc[0]['product']} ({kpi1.iloc[0]['units']} units)")
print(f"  KPI 2  │ Top Category  : {kpi2.iloc[0]['category']} (${kpi2.iloc[0]['revenue']:,.2f} revenue)")
print(f"  KPI 3  │ Most Abandoned: {kpi3.iloc[0]['product']} ({kpi3.iloc[0]['abandoned']}× abandoned)")
print(f"  KPI 4  │ Paying players: {n_paying}/580 ({pct_paying}%)")
print(f"  KPI 5  │ Top Whale     : {kpi5.iloc[0]['username']} — ${kpi5.iloc[0]['spent']:.0f}")
print(f"  KPI 6  │ Highest Rev   : {kpi6.iloc[0]['rank']} rank (avg ${kpi6.iloc[0]['avg_spent']:.2f})")
print(f"{'─'*65}")
print(f"  💰 Actual Revenue     : ${total_revenue:>9,.2f}")
print(f"  🔴 Abandoned Cart Val : ${total_abandoned:>9,.2f}  ← EXCEEDS revenue!")
print(f"  📉 Revenue Capture    : {capture_rate:.1f}%  (losing {100-capture_rate:.1f}% to abandonment)")
print(f"  🚀 50% Recovery = +${total_abandoned*0.5:,.0f} extra revenue")
print(f"{'═'*65}")


<a id="part-a"></a>
## Part A — Player Psychology: Breaking the Purchasing Barrier

**Business question:** How long does it take a new player to make their first real purchase?
Understanding this *psychological barrier* lets us time promotional nudges precisely.

**Approach:**
- Filter to paying users who have a recorded `last_purchase_date`.
- Calculate `days_to_purchase = last_purchase_date − first_join_date`.
- Bucket players into conversion-speed segments.
- Break down median conversion speed by rank.

> **Note:** Because the dataset captures `last_purchase_date` rather than a
> dedicated first-purchase field, single-transaction users give us an exact
> first-purchase gap, while multi-transaction users give us a conservative
> upper bound. This makes the "fast converters" analysis more reliable than
> the tail.

In [ ]:
# -- Data prep -------------------------------------------------------------
payers = players[players['is_payer'] & players['last_purchase_date'].notna()].copy()
payers['days_to_purchase'] = (payers['last_purchase_date'] - payers['first_join_date']).dt.days
payers = payers[payers['days_to_purchase'] >= 0]

BIN_EDGES  = [-1, 0, 7, 30, 90, 365, 99_999]
BIN_LABELS = ['Same Day', '1-7 Days', '8-30 Days', '31-90 Days', '91-365 Days', '> 1 Year']
payers['time_bucket'] = pd.cut(payers['days_to_purchase'], bins=BIN_EDGES, labels=BIN_LABELS)
bucket_counts  = payers['time_bucket'].value_counts().reindex(BIN_LABELS, fill_value=0)
bucket_pct     = (bucket_counts / bucket_counts.sum() * 100).round(1)
median_by_rank = payers.groupby('rank')['days_to_purchase'].median().sort_values()
overall_median = payers['days_to_purchase'].median()

fig = plt.figure(figsize=(26, 10), facecolor=C['dark_bg'])
fig.suptitle('Part A  —  Player Psychology: Breaking the Purchasing Barrier',
             fontsize=18, fontweight='bold', color=C['text'], y=1.02)
ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# Histogram
clipped = payers['days_to_purchase'].clip(upper=730)
ax1.hist(clipped, bins=40, color=C['green'], edgecolor='#000', alpha=0.85)
ax1.axvline(overall_median, color=C['gold'], lw=2.5, label=f'Median: {overall_median:.0f} days')
ax1.axvline(30, color=C['orange'], lw=1.5, linestyle=':', alpha=0.7, label='30-day mark')
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax1, 'Days from Join → Last Purchase\n(capped at 730 days)', 'Days', 'Number of Players')

# Bucket bars
seg_colors = [C['green'], C['teal'], C['blue'], C['purple'], C['orange'], C['red']]
bars2 = ax2.bar(BIN_LABELS, bucket_counts.values, color=seg_colors, edgecolor='#000', width=0.65)
for bar, pct, cnt in zip(bars2, bucket_pct, bucket_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{cnt}\n({pct}%)', ha='center', va='bottom', fontsize=11,
             color=C['text'], fontweight='bold', linespacing=1.4)
ax2.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax2, 'Conversion Speed Segments\nHow Fast Do Players First Purchase?',
      'Time Window', 'Players', rotate_x=True)

# Median by rank
r_colors = [RANK_COLORS.get(r, C['blue']) for r in median_by_rank.index]
bars3 = ax3.barh(median_by_rank.index, median_by_rank.values, color=r_colors, edgecolor='#000', height=0.5)
for bar in bars3:
    ax3.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.0f} days', va='center', fontsize=12,
             color=C['text'], fontweight='bold')
ax3.set_xlim(0, median_by_rank.max() * 1.28)
ax3.grid(axis='x', alpha=0.3, linewidth=0.8)
style(ax3, 'Median Days to Purchase by Rank\nWhich Tier Converts Fastest?', 'Median Days', '')

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('part_a_player_psychology.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

early = bucket_counts['Same Day'] + bucket_counts['1-7 Days'] + bucket_counts['8-30 Days']
print(f"★  Paying players analysed  : {len(payers):,}")
print(f"    Overall median           : {overall_median:.0f} days")
print(f"    Converted ≤ 30 days      : {early} ({bucket_pct['Same Day']+bucket_pct['1-7 Days']+bucket_pct['8-30 Days']:.1f}%)")
print(f"    Same-day converters      : {bucket_counts['Same Day']} ({bucket_pct['Same Day']}%)")
print(f"    Waited > 1 year          : {bucket_counts['> 1 Year']} ({bucket_pct['> 1 Year']}%)")


### Part A * Business Conclusions

| Finding | Implication |
|-----|-------|
| **Same-day converters are the largest single group.** A meaningful portion of paying players make their first purchase the same day they join. | Run a **"Welcome Bonus"** campaign - a 10-15 % first-purchase discount that expires 48 hours after account creation will capture more of these high-intent joiners before their excitement fades. |
| **The 1-30 day window captures the bulk of eventual buyers.** Players who haven't bought within a month are exponentially less likely to convert organically. | Trigger an automated email/Discord DM at **day 7** and **day 28** for webstore-visitors with 0 purchases. Offer a time-limited incentive (e.g., free Common Crate Key). |
| **Legend-rank players convert fastest.** Their shorter median time reflects either prior experience with the server or stronger disposable intent. | High-rank players are *aspirational buyers*. Pre-launch teaser posts ("MVP -> Legend upgrade only $25 this weekend") will accelerate their conversion window further. |
| **The > 1 Year tail is small but real.** Long-dormant accounts that eventually spend likely re-engaged via a hype event. | One **annual "Comeback Event"** (double-vote rewards, discounted ranks) can systematically monetise this tail. |


<a id="part-b"></a>
## Part B — Targeted Segments: Sleeper VIPs & Cart Abandonment

**Business question:** Who are the players we can move with a single targeted action?

Two high-yield segments:
1. **Sleeper VIPs** — `Default`-rank players with zero spend but top-quartile votes *and* playtime.
   They are deeply invested in the server but have never opened their wallets. One right offer can flip them.
2. **Cart Abandoners** — Players who repeatedly add items to cart but never check out.
   These are revealed purchase intent signals sitting as uncollected revenue.

In [ ]:
# -- Segment 1: Sleeper VIPs -----------------------------------------------
vote_q70     = players['total_votes'].quantile(0.70)
playtime_q70 = players['total_playtime_hours'].quantile(0.70)
sleepers = players[
    (players['rank'] == 'Default') & (~players['is_payer']) &
    (players['total_votes'] >= vote_q70) & (players['total_playtime_hours'] >= playtime_q70)
].sort_values('total_votes', ascending=False)

# -- Segment 2: Cart Abandonment by Rank -----------------------------------
rank_abandon = players.groupby('rank', observed=True).agg(
    total_abandonments=('cart_abandonments', 'sum'),
    avg_abandonments=('cart_abandonments', 'mean'),
    player_count=('id', 'count'),
    total_cart_value=('cart_total_value', 'sum'),
).reset_index().sort_values('total_abandonments', ascending=False)

high_abandoners = players[players['cart_abandonments'] >= 4].sort_values(
    'cart_abandonments', ascending=False
)[['username','rank','cart_abandonments','cart_total_value','webstore_visits','total_votes']]

fig, axes = plt.subplots(1, 3, figsize=(28, 10), facecolor=C['dark_bg'])
fig.suptitle('Part B  —  Targeted Segments: Sleeper VIPs & Cart Abandonment',
             fontsize=18, fontweight='bold', color=C['text'])

# Scatter: Sleeper VIPs
ax = axes[0]
bubble_size = sleepers['webstore_visits'] * 6 + 40
ax.scatter(sleepers['total_playtime_hours'], sleepers['total_votes'],
           s=bubble_size, c=C['green'], edgecolors=C['gold'], linewidths=1.5, alpha=0.8)
for _, row in sleepers.head(7).iterrows():
    ax.annotate(row['username'], (row['total_playtime_hours'], row['total_votes']),
                textcoords='offset points', xytext=(6, 2), fontsize=9, color=C['gold'])
ax.axvline(playtime_q70, color=C['orange'], lw=1, linestyle='-', alpha=0.5, label='P70 playtime')
ax.axhline(vote_q70,     color=C['blue'],   lw=1, linestyle='-', alpha=0.5, label='P70 votes')
ax.legend(fontsize=10)
ax.grid(alpha=0.2, linewidth=0.8)
style(ax, f'Sleeper VIPs — {len(sleepers)} Players\nDefault | $0 Spent | Top-30% Engagement',
      'Playtime (hours)', 'Total Votes')
ax.text(0.98, 0.02, 'Bubble = webstore visits', transform=ax.transAxes,
        ha='right', fontsize=9, color='#aaa')

# Grouped bar: abandonments by rank
ax2 = axes[1]
x, w = np.arange(len(rank_abandon)), 0.38
b1 = ax2.bar(x - w/2, rank_abandon['total_abandonments'], w,
             label='Total Abandonments', color=C['red'], edgecolor='#000', alpha=0.9)
b2 = ax2.bar(x + w/2, rank_abandon['avg_abandonments'], w,
             label='Avg / Player', color=C['orange'], edgecolor='#000', alpha=0.9)
ax2.set_xticks(x)
ax2.set_xticklabels(rank_abandon['rank'], fontsize=12, color=C['text'])
ax2.bar_label(b1, fmt='%d',   fontsize=10, color=C['text'], padding=3)
ax2.bar_label(b2, fmt='%.1f', fontsize=10, color=C['text'], padding=3)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax2, 'Cart Abandonments by Rank\nTotal Count vs. Per-Player Average', 'Rank', 'Count')

# Bar: abandoned cart $ by rank
ax3 = axes[2]
rank_colors_list = [RANK_COLORS.get(r, C['blue']) for r in rank_abandon['rank']]
bars3 = ax3.bar(rank_abandon['rank'], rank_abandon['total_cart_value'],
                color=rank_colors_list, edgecolor='#000', width=0.55)
for bar, (_, row) in zip(bars3, rank_abandon.iterrows()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f"${row['total_cart_value']:,.0f}", ha='center', va='bottom',
             fontsize=12, fontweight='bold', color=C['text'])
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax3, 'Abandoned Cart Value by Rank\nUSD Left Uncollected per Tier', 'Rank', 'USD ($)')

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('part_b_targeted_segments.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

print(f"★  Sleeper VIPs identified      : {len(sleepers)}")
print(f"    Their avg playtime            : {sleepers['total_playtime_hours'].mean():.0f} hrs")
print(f"    Their avg votes               : {sleepers['total_votes'].mean():.0f}")
print(f"\n⚠   Players with ≥4 abandonments : {len(high_abandoners)}")
print(f"    Total cart value (this group) : ${high_abandoners['cart_total_value'].sum():,.2f}")
print(f"\n{high_abandoners.head(10).to_string(index=False)}")


### Part B * Business Conclusions

| Finding | Implication |
|-----|-------|
| **Sleeper VIPs are the highest-ROI audience.** They already love the server (high votes + playtime) and have visited the webstore - they just haven't pulled the trigger. | Create a **"Loyal Player Reward"** bundle (VIP rank + 1 Legendary Crate Key) priced at a perceived discount. Send a personalised Discord DM to the top 20 sleepers by vote count. |
| **Default players generate the most total cart abandonments (1 061).** This is expected given volume (482 players) but their avg (2.20) is close to paid ranks, meaning even engaged non-payers drop carts at the same rate. | Implement a **cart reminder notification** (in-game `/msg` or Discord bot) 24 h after an abandoned session: *"You left [item] behind - grab it before stock runs out!"* Urgency framing drives the highest recovery rates. |
| **Legend players have the highest average abandonment rate (2.44).** They spend the most but also leave the most on the table. | High-rank players respond to **exclusivity**, not discounts. Show a **"Only 3 left at this price"** or **"Legend-exclusive bundle"** message to tip them past checkout friction. |
| **Repeat abandoners (? 4 events) are a micro-segment worth individual outreach.** | Flag these players in your CRM/Discord. A direct, personalised offer from a staff member converts far better than automated blasts. |


<a id="part-c"></a>
## Part C — Server Hype: Do Votes & Playtime Drive Revenue?

**Business question:** Does broader server engagement (votes on server-listing sites,
raw hours played) correlate with actual spending?
If yes, server-wide vote events and activity drives are justified marketing spend.

**Method:** Pearson correlation + OLS regression line for both `total_votes` and
`total_playtime_hours` against `total_spent_dollars` (paying players only).

In [ ]:
df_c = players[players['is_payer']].copy()

fig, axes = plt.subplots(1, 2, figsize=(24, 9), facecolor=C['dark_bg'])
fig.suptitle('Part C  —  Server Hype: Engagement Metrics vs. Revenue Correlation',
             fontsize=18, fontweight='bold', color=C['text'])

metrics = [
    ('total_votes',          'Total Votes',           C['purple']),
    ('total_playtime_hours', 'Total Playtime (hours)', C['teal']),
]
results = {}
for ax, (xcol, xlabel, dot_color) in zip(axes, metrics):
    x, y = df_c[xcol].values, df_c['total_spent_dollars'].values
    point_colors = [RANK_COLORS.get(r, dot_color) for r in df_c['rank']]
    ax.scatter(x, y, c=point_colors, s=50, alpha=0.60, edgecolors='none', zorder=3)

    slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 300)
    ax.plot(x_line, slope * x_line + intercept, color=C['gold'], lw=2.5, zorder=4,
            label=f'OLS: r = {r_val:.3f}')
    sig = '✔ Significant' if p_val < 0.05 else '✗ Not significant'
    ax.text(0.98, 0.05,
            f'r = {r_val:+.3f}\nR² = {r_val**2:.3f}\np = {p_val:.4f}\n{sig}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
            color=C['text'], fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', fc=C['card_bg'], ec=C['grid'], alpha=0.9))
    ax.legend(fontsize=11)
    ax.grid(alpha=0.2, linewidth=0.8)
    results[xcol] = (r_val, p_val)
    style(ax, f'{xlabel} vs. Revenue\nCorrelation Analysis (Paying Players Only)',
          xlabel, 'Total Spent ($)')

patches = [mpatches.Patch(color=v, label=k) for k, v in RANK_COLORS.items()]
axes[1].legend(handles=patches, fontsize=11, loc='upper left', title='Rank', title_fontsize=11)

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('part_c_server_hype.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

print("★  Correlation Results (paying players only):")
for col, label in [('total_votes','Votes'), ('total_playtime_hours','Playtime')]:
    r, p = results[col]
    print(f"    {label:12s} → Revenue | r = {r:+.3f}  p = {p:.4f}  [{'✔' if p<0.05 else '✗'} sig]")
print(f"\n    Avg spend by rank:")
for rank in ['Default','VIP','MVP','Legend']:
    s = df_c[df_c['rank']==rank]
    if len(s): print(f"    {rank:8s} : ${s['total_spent_dollars'].mean():.2f}  ({len(s)} players)")


### Part C * Business Conclusions

| Finding | Implication |
|-----|-------|
| **Votes correlate positively with revenue.** Players who vote on server-listing sites spend more - voting is a proxy for *active investment* in the server's success. | Budget 1-2 staff hours per month on a **"Vote Drive"** event (double-vote rewards for a weekend). Higher vote counts attract new organic traffic *and* signal deeper spending intent in existing players. |
| **Playtime correlation is weaker but real.** Extremely high-playtime players don't automatically become big spenders - they may be "grinders" who prefer to earn rather than buy. | Don't assume power-users will self-convert. Target **mid-range playtime players** (50-300 hrs) with purchase nudges - they have enough attachment to spend but haven't optimised out the need for purchased perks. |
| **Legend-rank players dominate the top-right quadrant.** High votes + high spend = your VIP advocates. | Identify your top 10 Legend spenders and recruit them as **unofficial brand ambassadors** (custom tag, private Discord channel). Word-of-mouth from respected players is your cheapest acquisition channel. |
| **Low-vote payers exist.** Some players spend despite minimal voting - they found the server via another channel. | Diversify traffic sources (YouTube partner, TikTok content creator programme). These players convert independently of the vote-loop. |


<a id="part-d"></a>
## Part D — Financials: Actual Revenue vs. Potential Lost Revenue

**Business question:** What is the financial scale of cart abandonment, and
which product categories are the biggest "leaks"?

**Calculation:**
- **Actual Revenue** = `SUM(total_spent_dollars)` across all players.
- **Abandoned Cart Value** = sum of the exact `price` field of every item
  currently sitting in every player's `cart_items_list`.
- **Revenue Capture Rate** = Actual ÷ (Actual + Lost).

In [ ]:
actual_rev  = players['total_spent_dollars'].sum()
lost_rev    = players['cart_total_value'].sum()
capture_pct = actual_rev / (actual_rev + lost_rev) * 100
diff        = lost_rev - actual_rev

cat_lost = (cart_flat.groupby('category')['price'].sum()
            .sort_values(ascending=False).reset_index()
            .rename(columns={'price': 'lost_value'}))
item_lost = (cart_flat.groupby('item').agg(qty=('price','count'), value=('price','sum'))
             .sort_values('value', ascending=False).reset_index())

fig = plt.figure(figsize=(26, 10), facecolor=C['dark_bg'])
fig.suptitle('Part D  —  Financials: Actual Revenue vs. Potential Lost Revenue',
             fontsize=18, fontweight='bold', color=C['text'])
ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# Bar: actual vs lost
bars1 = ax1.bar(['Actual\nRevenue', 'Abandoned\nCart Value'],
                [actual_rev, lost_rev], color=[C['green'], C['red']], edgecolor='#000', width=0.45)
ax1.bar_label(bars1, labels=[f'${actual_rev:,.0f}', f'${lost_rev:,.0f}'],
              fontsize=14, fontweight='bold', color=C['text'], padding=8)
ax1.set_ylim(0, max(actual_rev, lost_rev) * 1.3)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.grid(axis='y', alpha=0.3, linewidth=0.8)
ax1.text(0.5, 0.88, f'⚠ Abandoned EXCEEDS\nrevenue by ${diff:,.0f}!',
         transform=ax1.transAxes, ha='center', fontsize=12, color=C['red'], fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', fc='#1a0505', ec=C['red'], alpha=0.9))
style(ax1, 'Revenue Reality Check\nActual Earned vs. Cart Abandoned', '', 'USD ($)')

# Donut: capture rate
ax2.pie([capture_pct, 100-capture_pct],
        labels=[f'Captured\n{capture_pct:.1f}%', f'Lost\n{100-capture_pct:.1f}%'],
        colors=[C['green'], C['red']], startangle=90,
        wedgeprops={'width': 0.55, 'edgecolor': C['dark_bg'], 'linewidth': 2.5},
        textprops={'color': C['text'], 'fontsize': 13, 'fontweight': 'bold'})
ax2.set_title('Revenue Capture Rate\n% of Total Potential Collected',
              color=C['text'], fontsize=16, fontweight='bold', pad=14)
ax2.text(0, 0, f'{capture_pct:.0f}%\ncaptured', ha='center', va='center',
         fontsize=18, color=C['green'], fontweight='bold')

# Horizontal bar: lost value by category
if len(cat_lost):
    cat_col_d = [CAT_COLORS.get(c, C['blue']) for c in cat_lost['category']]
    bars3 = ax3.barh(cat_lost['category'][::-1], cat_lost['lost_value'][::-1],
                     color=cat_col_d[::-1], edgecolor='#000', height=0.55)
    for bar in bars3:
        ax3.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'${bar.get_width():,.0f}', va='center', fontsize=11,
                 fontweight='bold', color=C['text'])
    ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax3.grid(axis='x', alpha=0.3, linewidth=0.8)
    style(ax3, 'Abandoned Revenue by Category\nWhich Product Lines Leak the Most?', 'USD ($)', '')

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('part_d_financials.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

print(f"{'═'*52}")
print(f"  Actual Revenue        : ${actual_rev:>10,.2f}")
print(f"  Abandoned Cart Value  : ${lost_rev:>10,.2f}  ← exceeds revenue!")
print(f"  Revenue Capture Rate  : {capture_pct:>9.1f}%")
print(f"  50% Cart Recovery     : +${lost_rev*0.5:>9,.2f}")
print(f"{'═'*52}")
print(f"\nTop abandoned items by total value:")
print(item_lost.head(10).to_string(index=False))


### Part D * Business Conclusions

| Finding | Implication |
|-----|-------|
| **The abandoned cart pool is a significant fraction of actual revenue.** Recovering even 30-50 % of that value would materially move the P&L without acquiring a single new player. | Prioritise **checkout friction reduction**: fewer clicks to payment, saved payment methods, clear price display. Every second of checkout friction increases abandonment. |
| **Ranks and Perks likely dominate the abandoned value.** These are high-ticket items ($20-$50) where buyers need reassurance before committing. | Add a **"Trust Signal"** at checkout: player count currently online, a social-proof ticker ("12 players bought Legend this week"), and a 7-day satisfaction policy statement. |
| **Repeat abandoners are the most recoverable segment.** They've shown intent multiple times - they just need the right trigger. | A **retargeting drip** (3 Discord messages over 7 days: info -> scarcity -> discount) recovers this cohort at higher rates than one-shot blasts. Cap at one discount offer to protect margin. |
| **The gap between actual and potential revenue represents your product-market fit ceiling.** | If abandoned value continues growing faster than actual revenue, your pricing may be above perceived value. Run a **limited-time 20 % bundle discount** to A/B test price elasticity. |


<a id="part-e"></a>
## Part E — Growth & New Products: ARPU Opportunities

**Business question:** Which product categories convert best *today*, and
where should we invest in new SKUs to maximise Average Revenue Per User (ARPU)?

**Method:**
- Explode `purchased_items_list` into a flat fact table and join with `store_products`
  to obtain category-level revenue, volume, and unique-buyer counts.
- `Revenue per Unique Buyer` = category revenue ÷ distinct players who bought
  at least one item in that category — the best ARPU proxy we have.
- Combine insights to suggest specific new webstore additions.

In [ ]:
cat_stats = (
    purchases_flat.groupby('category', observed=True)
    .agg(revenue=('price','sum'), units_sold=('item','count'),
         unique_buyers=('player_id','nunique'), avg_price=('price','mean'))
    .reset_index()
)
cat_stats['revenue_per_buyer'] = (cat_stats['revenue'] / cat_stats['unique_buyers']).round(2)
cat_stats = cat_stats.sort_values('revenue', ascending=False)

item_stats = (purchases_flat.groupby(['category','item'], observed=True)
              .agg(revenue=('price','sum'), units=('price','count'))
              .reset_index().sort_values('revenue', ascending=False))

fig = plt.figure(figsize=(28, 10), facecolor=C['dark_bg'])
fig.suptitle('Part E  —  Growth Lens: Category Performance & ARPU Opportunities',
             fontsize=18, fontweight='bold', color=C['text'])
ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

cat_col_e = [CAT_COLORS.get(c, C['blue']) for c in cat_stats['category']]

bars1 = ax1.bar(cat_stats['category'], cat_stats['revenue'], color=cat_col_e, edgecolor='#000', width=0.6)
ax1.bar_label(bars1, labels=[f'${v:,.0f}' for v in cat_stats['revenue']],
              fontsize=11, color=C['text'], padding=4)
ax1.set_ylim(0, cat_stats['revenue'].max() * 1.3)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax1, 'Total Revenue by Category\nWhich Category Earns the Most?',
      'Category', 'Revenue ($)', rotate_x=True)

bars2 = ax2.bar(cat_stats['category'], cat_stats['units_sold'], color=cat_col_e, edgecolor='#000', width=0.6)
ax2.bar_label(bars2, fmt='%d', fontsize=11, color=C['text'], padding=4)
ax2.set_ylim(0, cat_stats['units_sold'].max() * 1.3)
ax2.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax2, 'Units Sold by Category\nVolume vs. Revenue Trade-Off',
      'Category', 'Units Sold', rotate_x=True)

cat_arpu = cat_stats.sort_values('revenue_per_buyer', ascending=False)
arpu_col = [CAT_COLORS.get(c, C['blue']) for c in cat_arpu['category']]
bars3 = ax3.bar(cat_arpu['category'], cat_arpu['revenue_per_buyer'], color=arpu_col, edgecolor='#000', width=0.6)
ax3.bar_label(bars3, labels=[f'${v:.1f}' for v in cat_arpu['revenue_per_buyer']],
              fontsize=11, color=C['text'], padding=4)
ax3.set_ylim(0, cat_arpu['revenue_per_buyer'].max() * 1.3)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
ax3.grid(axis='y', alpha=0.3, linewidth=0.8)
style(ax3, 'Revenue per Unique Buyer (ARPU Proxy)\nWhere to Focus Premium Upsells?',
      'Category', 'Avg Revenue / Buyer ($)', rotate_x=True)

plt.tight_layout(pad=2.5, w_pad=3)
plt.savefig('part_e_growth.png', dpi=200, bbox_inches='tight', facecolor=C['dark_bg'])
plt.show()

print("★  Category Performance Summary:")
print(cat_stats[['category','revenue','units_sold','unique_buyers','avg_price','revenue_per_buyer']].to_string(index=False))
print("\n★  Top 5 best-selling individual items:")
print(item_stats.head(5)[['category','item','revenue','units']].to_string(index=False))
print("\n★  New product suggestions:")
suggestions = [
    ("Crate keys", "Seasonal Key (Summer/Halloween)", "$7.50", "Fills Unique→Legendary gap; limited-time FOMO"),
    ("Perks",      "XP Booster (×2 XP, 7 days)",     "$12",   "Recurring weekly repurchase"),
    ("Gkits",      "Kit Skyblock Pro / Factions Starter", "$8-12", "Game-mode specificity lifts conversion"),
    ("Ranks",      "Elite Rank (above Legend)",        "$80",   "Opens new ceiling; pulls Legend buyers up"),
    ("Tags",       "Custom Tag (staff-reviewed)",      "$15",   "High perceived value, near-zero cost"),
    ("Sellwands",  "Tier5 Sellwand (×1.5 bonus)",     "$25",   "Natural upsell for 38 Tier4 owners"),
]
print(f"\n{'Category':<14} {'Product':<36} {'Price':<8} Rationale")
print('─'*90)
for cat, prod, price, why in suggestions:
    print(f"{cat:<14} {prod:<36} {price:<8} {why}")


### Part E * Business Conclusions

| Finding | Implication |
|-----|-------|
| **Ranks drive the most total revenue** despite a higher price point and lower unit volume. | Ranks are the *anchor* product. Every webstore page should surface rank benefits first. Run a **"Rank Up" campaign** once per quarter. |
| **Crate Keys lead in unit volume.** They are your highest-frequency, low-friction product. | Add **Seasonal Keys** ($7.50) to fill the Unique?Legendary gap and capture mid-budget players who won't spend $10 but will spend $7.50. Run key bundles (3 for $20). |
| **Perks have the highest ARPU of all categories.** Buyers of Fly, Unban, and Max Land size spend more per transaction than any other segment. | Perks are under-marketed. Add a **"Recommended for you"** section on the webstore home page showing the top 2 perks by player type. Introduce new perks (XP Booster, Nickname Colour) at $10-15 to widen the catalogue. |
| **Gkits are low-revenue but steady.** Kit oracle / mythic / pro sell consistently but don't dominate. | Introduce **game-mode-specific Gkits** (Factions Starter Kit, Skyblock Farming Kit) at $8-12. Specificity dramatically increases conversion because players see immediate, relevant value. |
| **Tags (Random Tag, $5) are the cheapest entry point.** | Add a **Custom Tag** ($15) as a premium upsell. Custom cosmetics have near-zero production cost and very high perceived value among younger Minecraft players. |
| **There is no "Elite" tier above Legend.** The price ceiling is $50. | Launch an **"Elite"** rank at $80-100 with exclusive cosmetics, priority server access, and a unique in-game title. Aspiration is free to create and extremely effective at pulling Legend players up one more rung. |


<a id="readme"></a>
## Step 3 — Automated README Generation

The cell below dynamically builds `README.md` by embedding live computed
statistics (actual revenue, top segment sizes, etc.) directly from the
DataFrames calculated in the analysis above — so the document stays accurate
every time you re-run the notebook.

In [ ]:
actual_rev_readme  = players['total_spent_dollars'].sum()
lost_rev_readme    = players['cart_total_value'].sum()
capture_pct_readme = actual_rev_readme / (actual_rev_readme + lost_rev_readme) * 100

sleepers_count = len(players[
    (players['rank'] == 'Default') & (~players['is_payer']) &
    (players['total_votes'] >= players['total_votes'].quantile(0.70)) &
    (players['total_playtime_hours'] >= players['total_playtime_hours'].quantile(0.70))
])
top_cat     = cat_stats.iloc[0]['category'] if 'cat_stats' in dir() else 'Perks'
top_cat_rev = cat_stats.iloc[0]['revenue']  if 'cat_stats' in dir() else 0

readme_content = f"""# Craftiverse Consumer Behaviour Analysis (EN)

> **Network:** Craftiverse Factions & Skyblock  
> **Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Key Financial Snapshot

| Metric | Value |
|---|---|
| Total Players | 580 |
| Paying Players | {players['is_payer'].sum()} ({players['is_payer'].mean()*100:.1f}%) |
| **Actual Revenue** | **${actual_rev_readme:,.2f}** |
| **Abandoned Cart Value** | **${lost_rev_readme:,.2f}** |
| Revenue Capture Rate | {capture_pct_readme:.1f}% |
| Avg Spend (payers) | ${players[players['is_payer']]['total_spent_dollars'].mean():.2f} |
| Sleeper VIPs | {sleepers_count} |

> ⚠ See README.md (Hebrew) for full KPI explanations and strategies.
> ⚠ See SERVER_STRATEGY.md (Hebrew) for complete business strategy.
"""

Path('README_EN.md').write_text(readme_content, encoding='utf-8')
print(f"★  README_EN.md written ({Path('README_EN.md').stat().st_size:,} bytes)")
print(f"    Actual Revenue   : ${actual_rev_readme:,.2f}")
print(f"    Abandoned Value  : ${lost_rev_readme:,.2f}")
print(f"    Capture Rate     : {capture_pct_readme:.1f}%")


<a id="powerbi"></a>
## Step 4 — PowerBI Export

Export three clean, flat CSV files ready for import into PowerBI (or any
BI tool). The JSON columns are fully unnested; dates are ISO strings;
all prices are in USD as floats.

| File | Grain | Primary key |
|---|----|-------|
| `dim_players.csv` | One row per player | `player_id` |
| `fact_purchases.csv` | One row per purchased item | `player_id` + `item` |
| `fact_abandoned_carts.csv` | One row per abandoned cart item | `player_id` + `item` |

In [ ]:
# -- dim_players -----------------------------------------------------------
dim_players = players.drop(columns=['purchased_items_list', 'cart_items_list']).copy()
dim_players['first_join_date']    = dim_players['first_join_date'].dt.strftime('%Y-%m-%d')
dim_players['last_purchase_date'] = dim_players['last_purchase_date'].dt.strftime('%Y-%m-%d')
dim_players = dim_players.rename(columns={'id': 'player_id'})

# -- fact_purchases --------------------------------------------------------
fact_purchases = purchases_flat.copy()
fact_purchases.insert(0, 'purchase_key',
                      range(1, len(fact_purchases) + 1))

# -- fact_abandoned_carts --------------------------------------------------
fact_abandoned_carts = cart_flat.copy()
fact_abandoned_carts.insert(0, 'cart_key',
                            range(1, len(fact_abandoned_carts) + 1))

# -- Export ----------------------------------------------------------------
exports = {
    'dim_players.csv':           dim_players,
    'fact_purchases.csv':        fact_purchases,
    'fact_abandoned_carts.csv':  fact_abandoned_carts,
}
for fname, df in exports.items():
    df.to_csv(fname, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel/PowerBI BOM
    print(f"?  {fname:<30} {len(df):>5,} rows  *  {Path(fname).stat().st_size:>7,} bytes")

print("\n?  PowerBI import tips:")
print("    1. Use 'Get Data -> Text/CSV' and set delimiter to comma.")
print("    2. In the data model, relate dim_players[player_id] -> fact_purchases[player_id]")
print("    3. Relate dim_players[player_id] -> fact_abandoned_carts[player_id]")
print("    4. Mark dim_players as the Dimension table (1-side of the 1:many).")
print("    5. Create a Date table and link first_join_date / last_purchase_date.")


<a id="revenue-growth"></a>
## Revenue Growth Summary — 5 Data-Backed Strategies

> **Bottom line:** The server currently captures only **43.1%** of potential revenue.
> Every strategy below is derived directly from the analysis above — no new players required.

---

### 1. Fix the Checkout Leak — Cart Recovery (~+$3,937/year)

![Financials](part_d_financials.png)

**What Part D shows:** Abandoned cart value ($13,125) exceeds actual revenue ($9,912.50). More than half of every dollar a player intends to spend never completes checkout.

**3-step recovery drip:**

| Day | Message | Goal |
|:---:|:---|:---|
| 1 | *"You left [item] behind — here's what it does for you."* | Re-engage with value |
| 3 | *"⚠ Only 5 slots at this price. 67 players bought this week."* | FOMO before discount |
| 7 | *"🎁 One-time 10% off — Code: COMEBACK10 — expires in 24 h."* | Close with incentive |

> **Rule:** Never lead with a discount. Players who respond to scarcity don't need one — and early discounts train players to wait.

---

### 2. Capture Players on Day 1 — Welcome Flow (~+$800–$1,200/year)

![Player Psychology](part_a_player_psychology.png)

**What Part A shows:** Same-day converters are the single largest purchasing group. Over 70% of all eventual buyers make their first purchase within 30 days. After 90 days, conversion probability drops sharply.

**Actions:**
- **"Welcome Bonus"** — 15% off any item, expires 48 hours after account creation.
- Automated Discord/in-game nudge at **Day 7** and **Day 28** for store visitors with 0 purchases.
- Tease Legend-rank perks early — Legend players convert the fastest of all rank groups.

---

### 3. Wake Up Sleeping Loyalists — Sleeper VIP Campaign (~+$800–$1,500/year)

![Targeted Segments](part_b_targeted_segments.png)

**What Part B shows:** A subset of Default-rank players sit in the top 30% for both votes and playtime but have never spent a dollar. They are the most engaged non-paying players on the server.

**Action:** Target players matching `rank = 'Default' AND total_spent = 0 AND votes ≥ P70 AND playtime ≥ P70` with a personalised Discord DM:

> *"Hey [username] — you've been one of our most loyal players. Here's a private offer: VIP Rank + Legendary Crate Key at 40% off. Valid 72 hours."*

Personalisation and exclusivity outperform broadcast discounts for this segment.

---

### 4. Turn Votes into Revenue — Vote Drive Campaigns (~+$1,500–$2,500/year)

![Server Hype](part_c_server_hype.png)

**What Part C shows:** Total votes have the strongest positive correlation with spending (Pearson r, p < 0.05). High-vote players cluster in the same quadrant as high-spend players. Heavy grinders (very high playtime) prefer earning over buying.

**Actions:**
- **"Vote Drive" weekends** once or twice a month — double vote rewards, Discord countdowns, 1–2 staff hours.
- Target **mid-range playtime players (50–300 hrs)** with purchase nudges — invested enough to spend, not yet in grind-mode.
- Recruit the **top 10 Legend spenders** as brand ambassadors — custom tag, private Discord channel.

---

### 5. Fill the Product Gaps — New SKUs (~+$1,900–$2,200/year)

![Growth](part_e_growth.png)

**What Part E shows:** Perks have the highest Revenue per Buyer of any category. Tags are underpriced. The Sellwand ladder has a hard ceiling at Tier 4 — 38 current owners have nowhere to upgrade.

| Category | New Product | Price | Rationale |
|:---|:---|:---|:---|
| Ranks | **Elite Rank** (above Legend) | $80 | Raises the ceiling; pulls Legend buyers upward |
| Perks | **XP Booster** (×2 XP, 7 days) | $12 | Weekly repurchase — recurring revenue |
| Gkits | **Skyblock / Factions Starter Kit** | $8–12 | Game-mode specificity lifts conversion |
| Crate keys | **Seasonal Key** (Summer / Halloween) | $7.50 | Fills gap between Common ($5) and Legendary ($10) |
| Tags | **Custom Tag** (staff-reviewed) | $15 | Near-zero cost, very high perceived value |
| Sellwands | **Tier5 Sellwand** (×1.5 bonus) | $25 | Natural upsell for 38 Tier4 owners |

---

### Revenue Potential Summary

| Strategy | Estimated Annual Uplift |
|:---|---:|
| Cart Recovery (30% conversion) | +$3,937 |
| Sleeper VIP Activation | +$800–$1,500 |
| Welcome Flow (15% coupon) | +$800–$1,200 |
| Bundle Strategy (AOV ×1.5) | +$1,500–$2,500 |
| Elite Rank (15 sales × $80) | +$1,200 |
| Seasonal Keys + New Products | +$700–$1,000 |
| **Total Potential** | **+$8,937–$11,337** |
| Current Revenue | $9,912.50 |
| **Full Potential (~×2)** | **~$19,000–$21,000** |